In [8]:
import pandas as pd

In [19]:
df = pd.read_csv(r"C:\Users\USER\OneDrive\Desktop\Enerji Tasarrufu Projesi\household_power_consumption.txt", sep=';')

C:\Users\USER\AppData\Local\Temp\ipykernel_9256\1707334668.py:1: DtypeWarning: Columns (2,3,4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"C:\Users\USER\OneDrive\Desktop\Enerji Tasarrufu Projesi\household_power_consumption.txt", sep=';')


In [10]:
df.head()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.0


In [11]:
print("Veri Boyutu:", df.shape)

Veri Boyutu: (2075259, 9)


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2075259 entries, 0 to 2075258
Data columns (total 9 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Date                   object 
 1   Time                   object 
 2   Global_active_power    float64
 3   Global_reactive_power  float64
 4   Voltage                float64
 5   Global_intensity       float64
 6   Sub_metering_1         float64
 7   Sub_metering_2         float64
 8   Sub_metering_3         float64
dtypes: float64(7), object(2)
memory usage: 142.5+ MB


In [17]:
import numpy as np

In [21]:
df.replace('?', np.nan, inplace=True) #Hatalı '?' karakterleri NaN yapıp veri tiplerini dönüştürme

sayisal_sutunlar = [
    'Global_active_power', 
    'Global_reactive_power', 
    'Voltage', 
    'Global_intensity', 
    'Sub_metering_1', 
    'Sub_metering_2', 
    'Sub_metering_3'
]

for sutun in sayisal_sutunlar: #Her bir sütunu float yapma
    df[sutun] = df[sutun].astype(float)

df_temiz = df.dropna().copy() #Eksik verisi olan satırları silme

df_temiz.info() #Temizleme sonrası veri tipini kontrol etme

<class 'pandas.core.frame.DataFrame'>
Index: 2049280 entries, 0 to 2075258
Data columns (total 9 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Date                   object 
 1   Time                   object 
 2   Global_active_power    float64
 3   Global_reactive_power  float64
 4   Voltage                float64
 5   Global_intensity       float64
 6   Sub_metering_1         float64
 7   Sub_metering_2         float64
 8   Sub_metering_3         float64
dtypes: float64(7), object(2)
memory usage: 156.3+ MB


In [22]:
#Date ve Time sütunlarını hızlı bir şekilde birleştirip gerçek tarih saat formatına çevirme
df_temiz['Tarih_Saat'] = pd.to_datetime(
    df_temiz['Date'] + ' ' + df_temiz['Time'], 
    dayfirst=True
)

#Saat bilgisini ayrı bir sütun olarak çıkarma(0-23 arası)
df_temiz['Saat'] = df_temiz['Tarih_Saat'].dt.hour

#Sadece tarih bilgisini çıkarma(Günlük özet almak için)
df_temiz['Tarih'] = df_temiz['Tarih_Saat'].dt.date

#İlk 5 satıra bakma
df_temiz[['Date', 'Time','Tarih_Saat', 'Saat', 'Tarih']].head()

,Date,Time,Tarih_Saat,Saat,Tarih
0,16/12/2006,17:24:00,2006-12-16 17:24:00,17,2006-12-16
1,16/12/2006,17:25:00,2006-12-16 17:25:00,17,2006-12-16
2,16/12/2006,17:26:00,2006-12-16 17:26:00,17,2006-12-16
3,16/12/2006,17:27:00,2006-12-16 17:27:00,17,2006-12-16
4,16/12/2006,17:28:00,2006-12-16 17:28:00,17,2006-12-16


In [23]:
# Gece ve gündüz saatlerini beliirleme (22.00-06.00 arası Gece=1, Gündüz= 0)
df_temiz['Gece_Mi'] = df_temiz['Saat'].apply(lambda x:1 if (x >= 22 or x < 6) else 0)

# Her gün için toplam tüketimi ve gece/gündüz ayrımını hesaplama
gunluk_ozet = df_temiz.groupby('Tarih').agg(
    Toplam_kWh=('Global_active_power', lambda x: x.sum() * (60/1000)),
    Gece_kWh=('Global_active_power', lambda x: x[df_temiz.loc[x.index, 'Gece_Mi'] == 1].sum() * (60/1000)),
    Gunduz_kWh=('Global_active_power', lambda x: x[df_temiz.loc[x.index, 'Gece_Mi'] == 0].sum() * (60/1000))
).reset_index()

# Gece ve Gündüz tüketim oranlarını bulma
gunluk_ozet['Gece_Orani'] = gunluk_ozet['Gece_kWh'] / gunluk_ozet['Toplam_kWh']
gunluk_ozet['Gunduz_Orani'] = gunluk_ozet['Gunduz_kWh'] / gunluk_ozet['Toplam_kWh']

# İlk 5 satır görüntüleme
gunluk_ozet.head()

,Tarih,Toplam_kWh,Gece_kWh,Gunduz_kWh,Gece_Orani,Gunduz_Orani
0,2006-12-16,72.55056,15.34224,57.20832,0.211470,0.788530
1,2006-12-17,203.42760,52.73844,150.68916,0.259249,0.740751
2,2006-12-18,132.22956,20.84388,111.38568,0.157634,0.842366
3,2006-12-19,99.97164,12.83004,87.14160,0.128337,0.871663
4,2006-12-20,133.54488,28.09920,105.44568,0.210410,0.789590


In [24]:
import numpy as np

In [26]:
# Ek verinin fiziksel ve demografik özelliklerinin eklenmesi
# Sonuçların her çalışmada değişmemesi(sabit kalması) için seed ayarlıyorum
np.random.seed(42)
n_haneler = len(gunluk_ozet)

# Tüketimi 0 ve 1 arasında oranlama metrekare ve kişi sayısını buna göre atama
tuketim_skor = (gunluk_ozet['Toplam_kWh'] - gunluk_ozet['Toplam_kWh'].min()) / (gunluk_ozet['Toplam_kWh'].max() - gunluk_ozet['Toplam_kWh'].min())

# Ev metrekaresi (tüketim arttıkça artar) 50 ile 250 metrekare arası sınırlandırma
gunluk_ozet['Ev_Metrekare'] = (60 + tuketim_skor * 140 + np.random.normal(0, 10, n_haneler)).astype(int).clip(60, 250)

# Kişi sayısı (tüketim arttıkça artar) 1 ile 6 kişi arası
gunluk_ozet['Kisi_Sayisi'] = (1 + tuketim_skor * 4 + np.random.normal(0, 0.5, n_haneler)).astype(int).clip(1, 6)

# Oda sayısı (metrekareye göre mantıklı hesaplama) her 35 metrekareye 1 oda düşecek şekilde ayarlandı
gunluk_ozet['Oda_Sayisi'] = (gunluk_ozet['Ev_Metrekare'] // 35).clip(1, 6)

# Priz Sayılarını belirleme (Oda sayılarına göre)
gunluk_ozet['Toplam_Priz_Sayisi'] = gunluk_ozet['Oda_Sayisi'] * 4 + np.random.randint(2, 6, n_haneler)
gunluk_ozet['Aktif_Priz_Sayisi'] = (gunluk_ozet['Toplam_Priz_Sayisi'] * np.random.uniform(0.4, 0.85, n_haneler)).astype(int)

# Kategordik özellikleri belirleme (Isınma, Soğutma, Bahçe)
gunluk_ozet['Isinma_Turu'] = np.random.choice(['Doğalgaz', 'Klima', 'Elektrikli Soba', 'Merkezi'], size=n_haneler, p=[0.5, 0.2, 0.1, 0.2])
gunluk_ozet['Sogutma_Turu'] = np.random.choice(['Vantilator', 'Klima', 'Yok'], size=n_haneler, p=[0.3, 0.5, 0.2])
# Sadece 140 metrekareden büyük evlerin bahçesi olabilir şeklinde kural ekliyorum
gunluk_ozet['Bahce_Var_Mi'] = np.where(gunluk_ozet['Ev_Metrekare'] > 140, np.random.choice(['Evet', 'Hayır'], p=[0.6, 0.4], size=n_haneler), 'Hayır')

# Her bir satırı(günü) proje için bağımsız bir hane gibi isimlendirme
gunluk_ozet['Hane_ID'] = [f"HANE_{i:04d}" for i in range(1, n_haneler + 1)]

# Yeni ek olarak hazırladığım veri seti kısımları
gunluk_ozet[['Hane_ID', 'Toplam_kWh', 'Ev_Metrekare', 'Kisi_Sayisi', 'Oda_Sayisi', 'Aktif_Priz_Sayisi', 'Isinma_Turu', 'Bahce_Var_Mi']].head()

,Hane_ID,Toplam_kWh,Ev_Metrekare,Kisi_Sayisi,Oda_Sayisi,Aktif_Priz_Sayisi,Isinma_Turu,Bahce_Var_Mi
0,HANE_0001,72.55056,100,1,2,9,Elektrikli Soba,Hayır
1,HANE_0002,203.42760,157,3,4,9,Klima,Evet
2,HANE_0003,132.22956,130,2,3,8,Doğalgaz,Hayır
3,HANE_0004,99.97164,123,2,3,8,Doğalgaz,Hayır
4,HANE_0005,133.54488,122,3,3,7,Doğalgaz,Hayır
